# Install

In [ ]:
%pip install geopandas shapely requests pyyaml pandas numpy

# Setup

In [46]:
import pandas as pd
import numpy as np
import yaml
import geopandas as gpd
from shapely.geometry import Point
import re

# Load mmc data

In [ ]:
path_to_file = "data/mmc-10.yaml"

In [2]:
with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

In [ ]:
df

# Load city locations

In [70]:
# Za določitev v kateri NUTS regiji je občina
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_60M_2024_4326_LEVL_3.geojson"
gdf = gpd.read_file(url)
slovenia_regions = gdf[gdf['NUTS_ID'].str.startswith('SI', na=False)]

def get_slovenia_region(row):
    try:
        # Create the point
        point = Point(row['lng'], row['lat'])
        
        # Iterate through regions
        for _, region in slovenia_regions.iterrows():
            if region['geometry'].contains(point):
                return pd.Series([region['NUTS_NAME'], region['NUTS_ID']])
    except Exception:
        pass

    return pd.Series([None, None])

def slovensko_sklanjanje(row):
    ime = row["naselje"]
    if not isinstance(ime, str) or not ime:
        return pd.Series([None, None])

    # Pomožna funkcija za obdelavo posamezne besede
    def sklanjaj_besedo(beseda):
        # 1. Množinska imena (Abitanti, v Abitantih)
        if beseda.endswith('i'):
            return beseda + 'v', beseda + 'h'
        
        # 2. Ženska imena (Ljubljana, v Ljubljani)
        if beseda.endswith('a'):
            osnova = beseda[:-1]
            return osnova + 'e', osnova + 'i'
        
        # 3. Srednji spol (Velenje, Celje, Trebelno)
        if beseda.endswith('e') or (beseda.endswith('o') and len(beseda) > 3):
            osnova = beseda[:-1]
            # Preverimo prevoj (C, Č, Ž, Š, J) - v rodilniku ni vpliva, v mestniku pa
            return osnova + 'a', osnova + 'u'

        # 4. Moški spol (Maribor, Mokronog, Gradec)
        # Reševanje izpadajočega polglasnika (enostaven algoritem za -ec)
        osnova = beseda
        if beseda.endswith('ec'):
            osnova = beseda[:-2] + 'c'
        elif beseda.endswith('el') and beseda != 'Velenje': # npr. Angel -> Angla
            osnova = beseda[:-2] + 'l'
            
        return osnova + 'a', osnova + 'u'

    # Razbijemo na dele (upoštevamo presledke in vezaje)
    # Regex razbije "Mokronog-Trebelno" na ['Mokronog', '-', 'Trebelno']
    deli = re.split(r'(\s+|-)', ime)
    
    rodilnik_deli = []
    mestnik_deli = []
    
    for del_imena in deli:
        if del_imena.strip() == '' or del_imena == '-':
            rodilnik_deli.append(del_imena)
            mestnik_deli.append(del_imena)
        else:
            r, m = sklanjaj_besedo(del_imena)
            rodilnik_deli.append(r)
            mestnik_deli.append(m)
            
    return pd.Series(["".join(rodilnik_deli), "".join(mestnik_deli)])

In [ ]:
# Za združitev občin z naselji
naselje_obcina_file = "data/naselje-obcina.csv"
obcina_geoloc_file = "data/obcina-geoloc.csv"

naselje_obcina = pd.read_csv(naselje_obcina_file, sep=";") 
obcina_geoloc = pd.read_csv(obcina_geoloc_file)

df_naselje_cord = pd.merge(
    naselje_obcina, 
    obcina_geoloc[['city', 'lat', 'lng']], 
    left_on='občina', 
    right_on='city', 
    how='left'
)
df_naselje_cord = df_naselje_cord.drop(columns=['city'])
df_naselje_cord = df_naselje_cord.drop(columns=['občina'])
df_naselje_cord = df_naselje_cord.dropna()

df_naselje_cord[["rodilnik", "mestnik"]] = df_naselje_cord.apply(slovensko_sklanjanje, axis=1)

# Poda regijo kordinati
df_naselje_cord[['region_name', 'region_id']] = df_naselje_cord.apply(get_slovenia_region, axis=1)

# Dropa random občine, ki so pre blizu borderja, da jim kordinati zajbavajo
df_naselje_cord = df_naselje_cord.dropna()

# Da so imena na začetku
cols = ["naselje", "rodilnik", "mestnik", "lat", "lng", "region_name", "region_id"]
df_naselje_cord = df_naselje_cord[cols]

In [78]:
print(df_naselje_cord.head())
print(df_naselje_cord.count())

     naselje    rodilnik     mestnik      lat      lng            region_name  \
0   Abitanti   Abitantiv   Abitantih  45.5500  13.7333          Obalno-kraška   
1    Adamovo     Adamova     Adamovu  45.8363  14.6377      Osrednjeslovenska   
3   Adlešiči   Adlešičiv   Adlešičih  45.5711  15.1889  Jugovzhodna Slovenija   
4  Adrijanci  Adrijanciv  Adrijancih  46.8050  16.2172               Pomurska   
5       Ajba        Ajbe        Ajbi  46.0880  13.6347                Goriška   

  region_id  
0     SI044  
1     SI041  
3     SI037  
4     SI031  
5     SI043  
naselje        4952
rodilnik       4952
mestnik        4952
lat            4952
lng            4952
region_name    4952
region_id      4952
dtype: int64
